# Initialize Silver Fact Sales

One-time historical initialization. **Do not run** if `silver.fact_sale` already exists. Requires a schema-enabled Lakehouse named `lh_retail_dev` attached as the default Lakehouse and a pre-created `silver` schema. Reading the notebook does not run it. This notebook uses a full historical snapshot, not the incremental feed.

The row counts are verified for this particular sample dataset. Adapt them if the source changes. Ensure capacity headroom before executing expensive Spark operations.

In [ ]:
from pyspark.sql import functions as F

TABLE_NAME = "silver.fact_sale"
BASE_PATH = "Files/bronze/wwi-raw-data"
MONEY_COLUMNS = ("TotalExcludingTax", "TaxAmount", "TotalIncludingTax")

def normalized_sales(path):
    df = spark.read.parquet(path)
    for name in MONEY_COLUMNS:
        df = df.withColumn(name, F.col(name).cast("decimal(38,6)"))
    return df

def check_keys(df, expected_count=None):
    result = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct("SaleKey").alias("distinct_keys"),
        F.sum(F.col("SaleKey").isNull().cast("long")).alias("null_keys"),
        F.sum(F.col("InvoiceDateKey").isNull().cast("long")).alias("null_dates"),
        F.sum(F.col("Quantity").isNull().cast("long")).alias("null_quantities"),
        F.sum((F.col("Quantity") <= 0).cast("long")).alias("nonpositive_quantities"),
        *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in MONEY_COLUMNS]
    ).first()
    print(result.asDict())
    if expected_count is not None and result["rows"] != expected_count:
        raise ValueError(f"Unexpected row count: {result['rows']}")
    if result["rows"] != result["distinct_keys"] or result["null_keys"]:
        raise ValueError("SaleKey must be unique and not null")
    for name in ("null_dates", "null_quantities", "nonpositive_quantities", *(f"null_{c}" for c in MONEY_COLUMNS)):
        if result[name]:
            raise ValueError(f"Data-quality check failed: {name}={result[name]}")


In [ ]:
EXPECTED_HISTORICAL_ROWS = 50150843
if spark.catalog.tableExists(TABLE_NAME):
    raise RuntimeError(f"{TABLE_NAME} already exists. Initial load is intentionally blocked.")

historical = normalized_sales(f"{BASE_PATH}/full/fact_sale")
check_keys(historical, EXPECTED_HISTORICAL_ROWS)

# Preserve raw Bronze; write a new managed Delta table only.
initial = (historical
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_record_source", F.lit("historical")))
(initial.write.format("delta").mode("errorifexists").saveAsTable(TABLE_NAME))
print("Historical write completed")


In [ ]:
assert spark.catalog.tableExists(TABLE_NAME)
actual = spark.table(TABLE_NAME).count()
assert actual == EXPECTED_HISTORICAL_ROWS, (actual, EXPECTED_HISTORICAL_ROWS)
print("Historical load verified:", actual)
spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}").select("version", "operation", "operationMetrics").show(5, truncate=False)
